In [ ]:
from solana.rpc.api import Client
from solders.keypair import Keypair
from solders.transaction import VersionedTransaction
from solders.message import to_bytes_versioned
import requests
import base64
import json

with open("keys.json") as f:
    keys = f.read()
keys = json.loads(keys)

helius_api = keys["Helius_API"]
helius_rpc = keys["Helius_RPC"]
private_key = keys["Private_Key"]

In [8]:
import defi


In [9]:
defi.INPUT_MINT

'So11111111111111111111111111111111111111112'

In [31]:
client = Client(HELIUS_RPC_URL)
keypair = Keypair.from_base58_string(PRIVATE_KEY)
wallet_pubkey = keypair.pubkey()

In [ ]:
def get_jupiter_quote(input_mint: str, output_mint: str, amount: int, slippage_bps: int) -> dict:
    """Fetch a swap quote from Jupiter API."""
    params = {
        "inputMint": input_mint,
        "outputMint": output_mint,
        "amount": amount,
        "slippageBps": slippage_bps,
        "restrictIntermediateTokens": "true",
        "onlyDirectRoutes ": "true"
    }
    response = requests.get(JUPITER_QUOTE_API, params=params)
    response.raise_for_status()
    return response.json()

def get_jupiter_swap_transaction(quote_response: dict, wallet: str) -> dict:
    """Get serialized swap transaction from Jupiter API."""
    payload = {
        "quoteResponse": quote_response,
        "userPublicKey": wallet,
        "wrapAndUnwrapSol": True,  # Automatically handle WSOL wrapping/unwrapping
        "dynamicComputeUnitLimit": True,
        "prioritizationFeeLamports": {
              "priorityLevelWithMaxLamports": {
                  "maxLamports": 20000000, # 0.02 Sol
                  "global": True,
                  "priorityLevel": "veryHigh"
              }
          }
    }
    response = requests.post(JUPITER_SWAP_API, json=payload)
    response.raise_for_status()
    return response.json()

In [ ]:
try:
    buy_coin = "7oZcbyRE8nCLkwJ3pZnyrz9c4f6ucdC6RjUmrTHUCRaP"
    sell_coin = "So11111111111111111111111111111111111111112"
    amount = 
    buy_amount = sol_to_lamp(0.01)
    slippage = slip_to_bps(20)

    quote = get_jupiter_quote(sell_coin, buy_coin, AMOUNT, slippage)

    swap_tx = get_jupiter_swap_transaction(quote, str(wallet_pubkey))
    swap_instruction = swap_tx["swapTransaction"]

    # Step 3: Decode and sign the transaction
    raw_tx = VersionedTransaction.from_bytes(base64.b64decode(swap_instruction))    
    signature = keypair.sign_message(to_bytes_versioned(raw_tx.message))
    signed_tx = VersionedTransaction.populate(raw_tx.message, [signature])
    encoded_tx = base64.b64encode(bytes(signed_tx)).decode('utf-8')

    # Step 4: Send the transaction
    headers = {"Content-Type": "application/json"}
    data = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "sendTransaction",
        "params": [
            encoded_tx,
            {"skipPreflight": True, "encoding": "base64"}
        ]
    }
    response = requests.post(HELIUS_RPC_URL, headers=headers, json=data)
    response.raise_for_status()
    tx_id = response.json().get("result")
    print(f"Transaction sent: https://solscan.io/tx/{tx_id}")

except Exception as e:
    print(f"Error: {str(e)}")

Transaction sent: https://solscan.io/tx/4PfVpBdm2mNSQBZEQyXn5fmVtZb3Cw3RmfNHY53MGR7zH85N74ouEe4HAMoj96VmbxW9KSvMAWLDtfPTPEtYZEmp


In [44]:
response.json()

{'jsonrpc': '2.0',
 'result': '4PfVpBdm2mNSQBZEQyXn5fmVtZb3Cw3RmfNHY53MGR7zH85N74ouEe4HAMoj96VmbxW9KSvMAWLDtfPTPEtYZEmp',
 'id': 1}

In [37]:
response.json()

{'jsonrpc': '2.0',
 'error': {'code': -32602,
  'message': 'Invalid Request: ErrorObject { code: InvalidParams, message: "Invalid Request: invalid base58 encoding: InvalidCharacter { character: \'l\', index: 4 }", data: None }'},
 'id': 1}

In [ ]:
from solders.signature import Signature
tx_signature = Signature.from_string(tx_id)
tx_info = client.get_transaction(tx_signature, max_supported_transaction_version=0)
tx_info

In [58]:
tx_info.value.transaction.meta.post_token_balances[0]

UiTransactionTokenBalance(
    UiTransactionTokenBalance {
        account_index: 3,
        mint: "7oZcbyRE8nCLkwJ3pZnyrz9c4f6ucdC6RjUmrTHUCRaP",
        ui_token_amount: UiTokenAmount {
            ui_amount: Some(
                3515692.470566,
            ),
            decimals: 6,
            amount: "3515692470566",
            ui_amount_string: "3515692.470566",
        },
        owner: Some(
            "BJApez4geZSGd18vbmXw37CGanJxhdmLMZdkN5kTW2P7",
        ),
        program_id: Some(
            "TokenkegQfeZyiNwAJbNbGKPFXCWuBvf9Ss623VQ5DA",
        ),
    },
)

In [ ]:
post_balances = tx_info.value.transaction.meta.post_token_balances
    for balance in post_balances:
        if balance.mint == output_mint and balance.owner == wallet_pubkey:
            # Convert amount to human-readable (divide by 10^decimals)
            decimals = token.get_mint_info().decimals
            amount = balance.ui_token_amount.ui_amount
            if amount is None:
                raise ValueError("Amount not found in transaction balances")
            return amount

In [ ]:
memecoin = ""
buy_amount = sol_to_lamp(0.01)
slippage = slip_to_bps(20)

try:
    # Get swap quote
    quote = get_jupiter_quote(INPUT_MINT, memecoin, AMOUNT, slippage)

    # Get swap transaction
    swap_tx = get_jupiter_swap_transaction(quote, str(wallet_pubkey))
    swap_instruction = swap_tx["swapTransaction"]

    # Step 3: Decode and sign the transaction
    raw_tx = VersionedTransaction.from_bytes(base64.b64decode(swap_instruction))    
    signature = keypair.sign_message(to_bytes_versioned(raw_tx.message))
    signed_tx = VersionedTransaction.populate(raw_tx.message, [signature])
    encoded_tx = base64.b64encode(bytes(signed_tx)).decode('utf-8')

    # Step 4: Send the transaction
    headers = {"Content-Type": "application/json"}
    data = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "sendTransaction",
        "params": [
            encoded_tx,
            {"skipPreflight": True}
        ]
    }
    response = requests.post(HELIUS_RPC_URL, headers=headers, json=data)
    response.raise_for_status()
    tx_id = response.json().get("result")
    print(f"Transaction sent: https://solscan.io/tx/{tx_id}")

except Exception as e:
    print(f"Error: {str(e)}")